In [9]:
import math
import time 

class Value:
    _count = 0

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label if label else f'v{Value._count}'
        Value._count += 1

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+', f'({self.label}+{other.label})')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*', f'({self.label}*{other.label})')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,), f'**{other}', f'({self.label}**{other})')

        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward

        return out

    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __radd__(self, other): return self + other

    def tanh(self):
        x = self.data
        t = (math.exp(2 * x) - 1) / (math.exp(2 * x) + 1)
        out = Value(t, (self,), 'tanh', f'tanh({self.label})')

        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward

        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self,), 'exp', f'exp({self.label})')

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward

        return out

    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


def trace_and_print_gradients(output_node):
    nodes = []
    visited = set()

    def build(v):
        if v not in visited:
            visited.add(v)
            for child in v._prev:
                build(child)
            nodes.append(v)

    build(output_node)

    print(f"{'Label':20s} | {'Data':>10s} | {'Grad':>10s}")
    print("-" * 50)
    for node in reversed(nodes):
        print(f"{node.label:20s} | {node.data:10.4f} | {node.grad:10.4f}")



x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
b = Value(6.8813735870195432, label='b')


o = (x1 * w1 + x2 * w2 + b).tanh()
o.label = 'o'


start = time.perf_counter()
o.backward()
end = time.perf_counter()


trace_and_print_gradients(o)
print(f"\nTime taken for backward propagation: {(end - start) * 1e6:.2f} µs")


Label                |       Data |       Grad
--------------------------------------------------
o                    |     0.7071 |     1.0000
(((x1*w1)+(x2*w2))+b) |     0.8814 |     0.5000
((x1*w1)+(x2*w2))    |    -6.0000 |     0.5000
(x1*w1)              |    -6.0000 |     0.5000
w1                   |    -3.0000 |     1.0000
x1                   |     2.0000 |    -1.5000
(x2*w2)              |     0.0000 |     0.5000
w2                   |     1.0000 |     0.0000
x2                   |     0.0000 |     0.5000
b                    |     6.8814 |     0.5000

Time taken for backward propagation: 88.70 µs


In [10]:
import torch
import time



# Inputs and weights
x1 = torch.tensor(2.0, requires_grad=True)
x2 = torch.tensor(0.0, requires_grad=True)
w1 = torch.tensor(-3.0, requires_grad=True)
w2 = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(6.8813735870195432, requires_grad=True)
# Forward pass
o = (x1 * w1 + x2 * w2 + b).tanh()
#o = torch.exp(n)

o.retain_grad()

# Timing the backward pass
start = time.perf_counter()
o.backward()
end = time.perf_counter()


# View gradients
print("Gradient of o:", o.grad)
print("Gradient of x1:", x1.grad)
print("Gradient of x2:", x2.grad)
print("Gradient of w1:", w1.grad)
print("Gradient of w2:", w2.grad)
print("Gradient of b:", b.grad)
print(f"\nTime taken to compute gradients: {(end - start) * 1e6:.2f} µs")


Gradient of o: tensor(1.)
Gradient of x1: tensor(-1.5000)
Gradient of x2: tensor(0.5000)
Gradient of w1: tensor(1.0000)
Gradient of w2: tensor(0.)
Gradient of b: tensor(0.5000)

Time taken to compute gradients: 642.90 µs


In [91]:
import random
class Neuron:
  
  def __init__(self, nin):
    self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
    self.b = Value(random.uniform(-1,1))
  
  def __call__(self, x):
    # w * x + b
    act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
    out = act.tanh()
    return out
  
  def parameters(self):
    return self.w + [self.b]

x=[Value(2.0), Value(3.0)]
n=Neuron(2)
n(x)

Value(data=-0.9848440350904216)

In [104]:
#this is one layer
class Layer:
  
  def __init__(self, nin, nout):
    self.neurons = [Neuron(nin) for _ in range(nout)]
  
  def __call__(self, x):
    outs = [n(x) for n in self.neurons]
    return outs[0] if len(outs) == 1 else outs
  
  def parameters(self):
    return [p for neuron in self.neurons for p in neuron.parameters()]



In [105]:
x=[Value(2.0), Value(3.0),Value(-1)]
n=Layer(3,4)
n(x)

[Value(data=0.992604288261465),
 Value(data=-0.9672577882587304),
 Value(data=-0.8128999806025794),
 Value(data=-0.2971162455642053)]

In [106]:
class MLP:
  
  def __init__(self, nin, nouts):
    sz = [nin] + nouts
    self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
  
  def __call__(self, x):
    for layer in self.layers:
      x = layer(x)
    return x
  
  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()]


In [107]:
x=[Value(2.0), Value(3.0),Value(-1)]
n=MLP(3,[4,4,1])
n(x)

Value(data=-0.7966690967496662)

In [109]:
# training data
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]  # desired targets

# training loop
for k in range(20):
    # forward pass
    ypred = [n(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred)) 

    for p in n.parameters():
        p.grad=0.0

    # backward pass
    loss.backward()

    # update
    for p in n.parameters():
        p.data += -0.05 * p.grad

    print(k, loss.data)


0 6.5001059819773115
1 5.236837896978823
2 3.9950530189786653
3 3.945724796349746
4 3.8980327347051094
5 3.8474912291632553
6 3.7905792506670504
7 3.722930775791399
8 3.6385648287358374
9 3.529469609112519
10 3.385473819068932
11 3.1948554740711437
12 2.945357975035601
13 2.6234620541055342
14 2.2142923290077112
15 1.7321376796338477
16 1.261844064328948
17 0.8826127467804554
18 0.6117337623220346
19 0.4324469013500567
